In [1]:
import geopandas as gpd
import pandas as pd

PARQUET_PATH = "/home/isalvador/git/ssmtg/data/inrix_202505"
DATA_DIR="synth_data"

network = gpd.read_file(f"{DATA_DIR}/chicago_osm.shp")
vehicle_highway_types = [
    'motorway', 'motorway_link',
    'trunk', 'trunk_link',
    'primary', 'primary_link',
    'secondary', 'secondary_link',
    'tertiary', 'tertiary_link',
    'unclassified',
    'residential',
    'living_street',
    'service',
    'busway',
]
network_vehicle = network[network['highway'].isin(vehicle_highway_types)].copy()
raw_trajectories = pd.read_parquet(PARQUET_PATH)

In [2]:
import duckdb
import numpy as np
import time

t0 = time.time()

# ============================================================
# STEP 1 — Derive u/v node ids from network (small, stays in pandas/geopandas)
# ============================================================
print("Step 1: Deriving u/v node ids from network geometry...")

coords_start = np.array([geom.coords[0] for geom in network_vehicle.geometry])
coords_end = np.array([geom.coords[-1] for geom in network_vehicle.geometry])

# encode node id as a string key instead of a tuple — much friendlier for SQL joins
network_vehicle['u'] = [f"{x:.6f}_{y:.6f}" for x, y in np.round(coords_start, 6)]
network_vehicle['v'] = [f"{x:.6f}_{y:.6f}" for x, y in np.round(coords_end, 6)]

edges = network_vehicle[['osm_id', 'u', 'v']]
print(f"  -> {len(edges)} network edges ({time.time() - t0:.2f}s)")

# ============================================================
# STEP 2 — Run the whole pipeline in DuckDB against the parquet file directly
# ============================================================
print("Step 2: Running SQL pipeline in DuckDB...")
t0 = time.time()

con = duckdb.connect()
con.register('edges', edges)

query = f"""
WITH traj AS (
    SELECT
        t.trip_id,
        t.traj_idx,
        t.way_id,
        t.way_idx,
        t.trip_start_utc_ts,
        t.trip_end_utc_ts,
        t.start_utc_ts,
        t.end_utc_ts,
        e.u,
        e.v
    FROM parquet_scan('{PARQUET_PATH}/*.parquet') t
    INNER JOIN edges e ON t.way_id = e.osm_id
),
with_prev AS (
    SELECT *,
        LAG(u) OVER w AS prev_u,
        LAG(v) OVER w AS prev_v
    FROM traj
    WINDOW w AS (PARTITION BY trip_id, traj_idx ORDER BY way_idx)
),
with_runs AS (
    SELECT *,
        SUM(CASE WHEN prev_u IS NULL
                   OR (u != prev_u AND u != prev_v AND v != prev_u AND v != prev_v)
                 THEN 1 ELSE 0 END)
            OVER (PARTITION BY trip_id, traj_idx ORDER BY way_idx
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS run_id
    FROM with_prev
),
run_lengths AS (
    SELECT *,
        COUNT(*) OVER (PARTITION BY trip_id, traj_idx, run_id) AS run_len
    FROM with_runs
),
with_max AS (
    SELECT *,
        MAX(run_len) OVER (PARTITION BY trip_id, traj_idx) AS max_run_len
    FROM run_lengths
),
best_run AS (
    SELECT *,
        MIN(CASE WHEN run_len = max_run_len THEN run_id END)
            OVER (PARTITION BY trip_id, traj_idx) AS chosen_run_id
    FROM with_max
)
SELECT
    trip_id, traj_idx, way_id, way_idx,
    trip_start_utc_ts, trip_end_utc_ts, start_utc_ts, end_utc_ts,
    ROW_NUMBER() OVER (PARTITION BY trip_id, traj_idx ORDER BY way_idx) - 1 AS path_idx
FROM best_run
WHERE run_id = chosen_run_id
ORDER BY trip_id, traj_idx, way_idx
"""

filtered = con.execute(query).df()
print(f"  -> {len(filtered)} rows kept ({time.time() - t0:.2f}s)")

filtered = con.execute(query).df()
print(f"  -> {len(filtered)} rows kept ({time.time() - t0:.2f}s)")

# ============================================================
# STEP 3 — Sanity check
# ============================================================
print("Step 3: Final summary")
n_traj_after = filtered[['trip_id', 'traj_idx']].drop_duplicates().shape[0]
print(f"  Trajectories after filtering: {n_traj_after}")
print(f"  Filtered rows: {len(filtered)}")

Step 1: Deriving u/v node ids from network geometry...
  -> 92265 network edges (2.26s)
Step 2: Running SQL pipeline in DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 28836273 rows kept (44.92s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 28836273 rows kept (89.95s)
Step 3: Final summary
  Trajectories after filtering: 1650972
  Filtered rows: 28836273


In [3]:
filtered.to_parquet("validation/filtered_trips.parquet")

In [4]:
filtered.shape

(28836273, 9)

In [9]:
filtered.trip_id.nunique()

1645560